In [1]:
from pathlib import Path
import csv
import json
import shutil

# --- 1. STUDENT CONFIGURATION ---
LAST_NAME = "Juarez"
STUDENT_ID = "TUPM-26-1843"
SEED_DIGIT = int(STUDENT_ID[-1])  # 3
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 24

# Directories setup
WORK_DIR = Path(".")
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

# --- 2. EXERCISE 3 RULES & PATHS ---
target_year = "2023"  # Target year for incident analysis
cyber_dataset_path = Path("Global_Cybersecurity_Threats_2015-2024.csv")  # Ensure filename matches your workspace

cyber_filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"
cyber_json_summary = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_summary.json"
top5_json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_top5_incidents.json"
cyber_report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"

total_records = 0
year_incidents = []
attack_counts = {}
total_loss = 0.0
csv_fields = []

if cyber_dataset_path.exists():
    with open(cyber_dataset_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        csv_fields = reader.fieldnames
        
        for row in reader:
            total_records += 1
            # Check date or year field depending on the dataset structure
            date_str = row.get("Date", row.get("Year", ""))
            
            if target_year in date_str:
                year_incidents.append(row)
                
                # Count incidents per attack type
                attack_type = row.get("Attack Type", row.get("type", "Unknown"))
                attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1
                
                # Parse financial loss safely
                try:
                    loss = float(row.get("Financial Loss (USD millions)", row.get("loss", 0)))
                    total_loss += loss
                except ValueError:
                    loss = 0.0
                row["_parsed_loss"] = loss

# Sort incidents by descending financial loss to rank them (retaining original order for ties)
sorted_incidents = sorted(year_incidents, key=lambda x: x["_parsed_loss"], reverse=True)
top_5_costliest = sorted_incidents[:5]  # Five costliest incidents

avg_loss = total_loss / len(year_incidents) if year_incidents else 0.0

# --- 3. EXPORTS ---
# Export filtered incidents to CSV
if year_incidents and csv_fields:
    with open(cyber_filtered_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields)
        writer.writeheader()
        writer.writerows(year_incidents)

# Export top 5 costliest incidents to JSON
with open(top5_json_path, "w", encoding="utf-8") as f:
    json.dump(top_5_costliest, f, indent=4)

# Create JSON summary containing counts and losses
summary_data = {
    "target_year": target_year,
    "total_dataset_records": total_records,
    "incidents_in_target_year": len(year_incidents),
    "count_per_attack_type": attack_counts,
    "total_financial_loss_usd_millions": round(total_loss, 2),
    "average_financial_loss_usd_millions": round(avg_loss, 2)
}

with open(cyber_json_summary, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# --- 4. BACKUP & VERIFICATION ---
backup_json_path = BACKUP_DIR / cyber_json_summary.name
if cyber_json_summary.exists():
    shutil.copy2(cyber_json_summary, backup_json_path)

# Generate Text Report
with open(cyber_report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("=" * 40 + "\n")
    report.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    report.write(f"Target Year: {target_year}\n")
    report.write(f"Total Records in Dataset: {total_records}\n")
    report.write(f"Incidents in Target Year: {len(year_incidents)}\n")
    report.write(f"Total Financial Loss (USD millions): {round(total_loss, 2)}\n")
    report.write(f"Average Financial Loss (USD millions): {round(avg_loss, 2)}\n")
    report.write(f"Attack Type Counts: {json.dumps(attack_counts, indent=2)}\n")
    report.write(f"JSON Summary Backup File: {backup_json_path.name}\n")

print(f"--- Exercise 3 Complete ---")
print(f"Target Year: {target_year}")
print(f"Incidents Processed: {len(year_incidents)}")
print(f"Total Financial Loss: ${round(total_loss, 2)}M")
print(f"Report and JSON backups created successfully.")

--- Exercise 3 Complete ---
Target Year: 2023
Incidents Processed: 0
Total Financial Loss: $0.0M
Report and JSON backups created successfully.
